In [1]:
### Scientific Computing
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

### IPython
from IPython.display import display

### Qiskit 
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

### Qiskit IBM Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, Executor, QuantumProgram
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.options_models.noise_learner_v3_options import NoiseLearnerV3Options
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

### Samplomatic
import samplomatic
from samplomatic import Twirl, InjectNoise, ChangeBasis, build
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions, get_annotation

### Qiskit Addons
from qiskit_addon_utils.exp_vals.measurement_bases import get_measurement_bases
from qiskit_addon_utils.exp_vals.expectation_values import executor_expectation_values
from qiskit_addon_utils.noise_management import trex_factors, gamma_from_noisy_boxes
from qiskit_addon_pna import generate_noise_mitigating_observable
from qiskit_addon_slc.bounds import compute_backward_bounds, compute_forward_bounds, compute_local_scales, merge_bounds
from qiskit_addon_slc.utils import generate_noise_model_paulis, map_modifier_ref_to_ref
from qiskit_addon_slc.visualization import draw_shaded_lightcone

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'qiskit'

## 2.3  The `InjectNoise` annotation and `NoiseLearnerV3`

In section 2.2, we showed that you can box a circuit automatically using the __boxing pass manager__. So far, we have only seen boxes annotated with the `Twirl` annotation. Pauli twirling turns each layer's coherent error into a _stochastic Pauli channel_. This is powerful because it makes the noise tractable. A sparse Pauli channel is a much simpler object than an arbitrary noise process. However, twirling alone does not remove the noise. 

In this section, we introduce the `InjectNoise` annotation. We will use this annotation, along with `NoiseLearnerV3` to learn the noise in each box. This will allow us to implement advanced error mitigation techniques such as __PNA__ and __SLC__ in Chapter 3. 

The section proceeds in three steps. First, in section 2.3.1, the boxing pass manager is extended so that every gate box carries an `InjectNoise` annotation — a declared slot keyed by a `ref` string. Then, in section 2.3.2,  `NoiseLearnerV3` runs on the backend and fills each slot with a learned Pauli-Lindblad model. Finally, in section 2.3.3, we inspect the returned noise model and see which error generators dominate the layer.

### 2.3.1  Declaring the slot: `InjectNoise` on each gate box

In the cell below, we begin by generating a new boxing pass manager that we name `noise_learning_boxing_pm`. We keep the options used in the boxing manager `twirl_only_pm` above and add two new `inject_noise_*` options so that every gate box gets an `InjectNoise` annotation alongside its `Twirl`. We then apply this boxing pass manager to the toy circuit. 

`find_unique_box_instructions` then extracts the *distinct* gate layers from the circuit. These are the layers `NoiseLearnerV3` will actually characterize, since equivalent boxes (up to single-qubit dressing) share the same noise model. This is particularly efficient for circuits with many equivalent boxes as the noise learner only has to learn the noise in the _unique_ layers. 

In [ ]:
noise_learning_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="no_modification",
)

boxed_toy = noise_learning_boxing_pm.run(raw_toy_isa)

unique_toy_layers = find_unique_box_instructions(
    boxed_toy,
    normalize_annotations=None,
    undress_boxes=True,
)


As before, this boxing pass manager produces two boxes: one for the CZ layer and one for the measurement. Now, the CZ gate box carries an `InjectNoise` (from `inject_noise_targets="gates"`) and a `Twirl` annotation, while the measurement box keeps just its `Twirl`. 

The option `inject_noise_strategy="no_modification"` means that all the equivalent boxes (from `find_unique_box_instructions`) are assigned an inject noise annotation with the _same_ ref. `ref`is a `string` which uniquely identifies the Pauli-Lindblad map from which to inject noise. We can think of this as assigning all identical boxes with the same label.

In Chapter 3 we will have to change this option to `inject_noise_strategy=uniform_modification` (PNA) and `inject_noise_strategy=individual_modification` (SLC). 

For more information on the boxing pass manager, see the documentation [here](https://qiskit.github.io/samplomatic/api/auto/samplomatic.transpiler.generate_boxing_pass_manager.html#samplomatic.transpiler.generate_boxing_pass_manager).

In [ ]:
for i, inst in enumerate(unique_toy_layers):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a else "(none)"
    print(f"Layer {i}: ref = {ref}")
    display(inst.operation.body.draw("mpl", fold=-1, idle_wires=False))

In [ ]:
In the cell above, we print the `ref` string for each unique layer and draw the circuit diagram of the corresponding layers. 

Each unique layer with an `InjectNoise` annotation carries its own `ref` string. We can see above that the CZ box has a `ref` but the measurement layer, which carries no `InjectNoise` annotation, does not. 

`ref` is the label that `NoiseLearnerV3` will use to report each unique boxes noise once learnt. The same `ref` label will be used by `InjectNoise` to find the learned noise model again at runtime. 

The slots are now declared but empty. The next subsection sets out *what* `NoiseLearnerV3` will write into them.

#### What `NoiseLearnerV3` learns

`NoiseLearnerV3` characterizes each layer as a sparse *Pauli–Lindblad* channel,

$$\Lambda \;=\; \exp(\mathcal{L}), \qquad \mathcal{L}(\rho) \;=\; \sum_k \lambda_k\,(P_k\,\rho\,P_k - \rho),$$

following the model formalized by van den Berg, Minev, Kandala & Temme in [*Probabilistic error cancellation with sparse Pauli–Lindblad models on noisy quantum processors* , Nat. Phys. **19**, 1116–1121 (2023)](https://www.nature.com/articles/s41567-023-02042-2). Here $\mathcal{L}$ is a superoperator (it acts on density matrices $\rho$), and $\Lambda = e^{\mathcal{L}}$ is its formal exponential. Because the generators $\{P_k\}$ are Hermitian Paulis ($P_k^\dagger = P_k$, $P_k^2 = I$), the dissipator reduces to the form on the right.

The set of generators is fixed by the layer's connectivity: each involved qubit contributes the three 1-body Paulis $X_i, Y_i, Z_i$, and each gate edge contributes the nine non-identity 2-body Paulis $\{XX, XY, \ldots, ZZ\}$. For example, a single CZ on qubits $(0,1)$ therefore has $2\times 3 + 9 = 15$ generators in its model. The rates of these generators are what define the structure of the noise in the layer and `NoiseLearnerV3` learns the value of these generator rates. In section 2.3.2, we will learn the rates for the toy circuit and in section 2.3.3 we will inspect them, gaining insight into the structure of the noise.

_Note_: The model does not capture:
- __coherent error__: we assumed that coherent error hase been twirled into stochastic Pauli noise 
- __cross-layer crosstalk__: any generator outside the layer's connectivity would not appear. E.g. the generator $X_0 X_2$ would not appear in any layer that does not have gates acting on qubits $q_0$ and $q_2$. 

The noise learner is configured by three options that trade accuracy for QPU time: `num_randomizations` (independent random benchmarking circuits per configuration), `shots_per_randomization` (shots per circuit), and `layer_pair_depths` (the depths $d$ at which the layer is repeated; the learner fits the resulting fidelity-vs-depth decay to extract each generator's rate).

### 2.3.2  Run noise learning job on toy circuit

With the layers boxed up and declared, we send them to the backend. For this first toy circuit example, we choose small settings (`num_randomizations=5`, `shots_per_randomization=20`, `layer_pair_depths=[1, 2]`) so the QPU time stays short. 

_Note:_
The following cell executes a job on quantum hardware. Ensure you are ready to do this before proceeding. After first execution, we recommend pasting the job id into the `TOY_NOISE_LEARN_JOB_ID` parameter and setting `SUBMIT_TOY_NOISE_JOB = False`. This will prevent accidentally resubmitting the job and keeps the notebook re-runnable after a kernel restart.

_Estimated QPU execution time is 3 seconds (tested on ibm_pittsburgh)._ The usage estimate above reflects backend execution time only. Queue time, calibration, and runtime session delays may be longer.

In [ ]:
TOY_NOISE_LEARN_JOB_ID = None   # paste job_id here on re-run
SUBMIT_TOY_NOISE_JOB   = False   # set True to submit a fresh learning job

nl_options = NoiseLearnerV3Options(
    num_randomizations=5,
    shots_per_randomization=20,
    layer_pair_depths=[1, 2],
)
learner = NoiseLearnerV3(backend, nl_options)
learner.options.environment.job_tags = ["qgss26"]

if TOY_NOISE_LEARN_JOB_ID is not None:
    learner_job = service.job(TOY_NOISE_LEARN_JOB_ID)
    print(f"Re-using saved job: {TOY_NOISE_LEARN_JOB_ID}")
elif SUBMIT_TOY_NOISE_JOB:
    learner_job = learner.run(unique_toy_layers)
    TOY_NOISE_LEARN_JOB_ID = learner_job.job_id()
    print(f"Submitted: {TOY_NOISE_LEARN_JOB_ID}")
else:
    print("Set SUBMIT_TOY_NOISE_JOB=True to submit a fresh job, "
          "or paste a saved job id into TOY_NOISE_LEARN_JOB_ID and re-run.")



After submitting your noise learning job, run the cell below to check on the status of your job. It should either be: `QUEUED`, `RUNNING` or `DONE`. Once the job is done, proceed to the next cell. 

In [ ]:
learner_job = service.job(TOY_NOISE_LEARN_JOB_ID)
print(f"{TOY_NOISE_LEARN_JOB_ID}  (status: {learner_job.status()})")

In [ ]:
if learner_job.status() == "DONE":
    toy_noise_result = learner_job.result()
else:
    print(f"Not done yet (status={learner_job.status()}). Re-run cell when DONE.")

In [ ]:
if 'toy_noise_result' in dir() and toy_noise_result is not None:
    toy_refs_to_noise = toy_noise_result.to_dict(unique_toy_layers, require_refs=False)
    print(f"toy_refs_to_noise has {len(toy_refs_to_noise)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


In the cell above, we use the `to_dict` method to collect the results of the noise learning job into `toy_refs_to_noise`. There should be one Pauli-Lindblad model per learned layer, each labelled by the same `ref`s used in the boxed circuit. We know the toy circuit has only one layer with the `InjectNoise` annotation (the CZ layer). So we expect the noise learning job to return a single learned noise model. We see this is the case.

The slots are now full. To see what was actually learned, in section 2.3.3, we will inspect the learned noise model for the single CZ layer.

### 2.3.3  Inspecting the learned noise model

In the cell below, we use the method `PauliLindbladMap.to_sparse_list()` to return the noise model as a sequence of `(pauli, qubits, rate)` tuples. There is one tuple per nonzero generator of $\mathcal{L}$. 

We sort these tuples by rate to show which error channels dominate the CZ layer and print the top 10 generators. This information is the input every per-layer error mitigation strategy uses to decide where to spend its budget. we will see how this is implemented for PNA, PEC, and SLC in Chapter 3. 

In [ ]:
if "toy_refs_to_noise" not in globals() or not toy_refs_to_noise:
    print("No learned toy noise model yet. Run the noise-learning result cell above first.")
else:
    ref0, plm0 = next(iter(toy_refs_to_noise.items()))
    generators = plm0.to_sparse_list()
    generators.sort(key=lambda g: -abs(g[2]))

    print(f"Layer ref = {ref0}, total generators = {len(generators)}\n")
    print(f"  {'Pauli':<6}{'qubits':<14}{'rate':>10}")
    print(f"  {'-'*6}{'-'*14}{'-'*10}")
    for pauli, qubits, rate in generators[:10]:
        print(f"  {pauli:<6}{str(qubits):<14}{rate:>10.4e}")


One annotation still remains to introduce: `ChangeBasis`, for measurements in non-$Z$ bases. We do that next in section 2.4.

## 2.4  The `ChangeBasis` annotation

`ChangeBasis` is needed for measurements in non-$Z$ bases. It follows the same pattern as `InjectNoise`: the annotation is attached statically and supplied at runtime. It goes on a measurement box, and its values are bound at runtime through the samplex's `basis_changes` input. 

We don't make use of `ChangeBasis` in the rest of Chapter 2 as the observable we consider in section 2.6 is diagonal in the computational basis. But, in Chapter 3 we use it once PNA introduces a noise mitigating observable $\tilde{O}$ with non-$Z$ terms. 

The cell below shows that `measure_annotations="all"` adds both `Twirl` and `ChangeBasis` to the measurement box, which adds a `basis_changes.<ref>` slot to `samplex.inputs()`.

In [ ]:
# A boxing pass manager that ALSO annotates the measurement box with
# ChangeBasis. 
change_basis_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="no_modification",
)

demo_boxed = change_basis_pm.run(raw_toy_isa)
_, demo_samplex = build(demo_boxed)
print(demo_samplex)

The samplex now reports two **Inputs** instead of none as we saw in section 2.1.2 when there were only `Twirl` annotations on the boxes. The two inputs are:
- `basis_changes.basisX` (where X is an integer): the basis rotation that occurs per-randomization to be applied at measurement. This is encoded symplectically — `I=0, Z=1, X=2, Y=3`
- `pauli_lindblad_maps.<ref>`: the learned noise model for the gate box

Adding `ChangeBasis` and `InjectNoise` to the circuit changed what data the samplex expects to receive at run time. The `ref` strings (of the general form `basisX`, `rYYYY` where X is an integer and Y is an integer or letter) are the dictionary keys we will use to supply that data. 

The **Outputs** gain `pauli_signs` as well: with `InjectNoise` present, each randomization samples a Pauli error from the noise model, and the parity of negative-rate factors becomes a $\pm 1$ correction that the post-processing applies to expectation values later. 

So the same pattern carries through here: annotations *declare* what is needed (e.g. basis change, noise model, twirl), and the samplex *exposes* the slots and reports the runtime-bound results. Section 2.5 supplies the values for these slots and runs the program with `Executor`.

## 2.5  Submitting a job with the `Executor`

Now, we have learned the noise in the circuit, we submit a job using the [__Executor primitive__](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-executor). `Executor` is the Runtime primitive that honors Samplomatic annotations. 

As a quick refresher, remember that the __Sampler__ and __Estimator__ primitives take in a [Primitive Unified Bloc (PUB)](https://quantum.cloud.ibm.com/docs/en/guides/primitive-input-output#pubs). PUBs are tuples of which `QuantumCircuit`'s, parameters and for the Estimator, observables, are some of their inputs.

The inputs and outputs of the Executor primitive are very different from those of the Sampler and Estimator primitives. Instead of taking a list of PUBs as the input, Executor takes a `QuantumProgram`, which contains a list of `QuantumProgramItem` objects. These container classes allow for more flexibility than a PUB, which is a simple tuple data structure. These `QuantumProgramItem`s can either be a:
- `CircuitItem` which stores a circuit and its parameter values
- `SamplexItem` which stores a template circuit, a samplex and samplex arguments

Here, we are focussing on an Executor job that uses `SamplexItem`'s. First, we use `build` to build the template and samplex for the toy circuit:


In [ ]:
toy_template, toy_samplex = build(boxed_toy)

print(toy_samplex)

Next, we construct the samplex arguments. The keys of `samplex_arguments` must match those returned by `samplex.inputs()` exactly, including the auto-generated ref names like `noise_scales.<ref>` or `basis_changes.<ref>`.

In [ ]:
samplex_arguments = (
    toy_samplex.inputs()
    .make_broadcastable()
    .bind(pauli_lindblad_maps=toy_refs_to_noise)
)

Now we can construct the `QuantumProgram`. We use the method `append_samplex_item(...)` to add a `SamplexItem` to the `QuantumProgram` (more than one can coexist, which is how batched mitigated jobs are built). 

In [ ]:
program = QuantumProgram(shots=64)
program.append_samplex_item(
    toy_template,
    samplex=toy_samplex,
    samplex_arguments=samplex_arguments,
    shape=(16,),
)

We are now ready to run the Executor job on the QPU.

_Note:_ The following cell executes a job on quantum hardware. Ensure you are ready to do this before proceeding. After first execution, we recommend pasting the job id into the `TOY_EXECUTOR_JOB_ID` parameter and setting `SUBMIT_TOY_EXEC_JOB = False`. This will prevent accidentally resubmitting the job and keeps the notebook re-runnable after a kernel restart.

_Estimated QPU execution time is 2 seconds (tested on ibm_fez)._ The usage estimate reflects backend execution time only. Queue time, calibration, and runtime session delays may be longer.

In [ ]:
TOY_EXECUTOR_JOB_ID = None # paste job_id here on re-run
SUBMIT_TOY_EXEC_JOB = False # set True to submit a fresh executor job

executor = Executor(backend)
executor.options.environment.job_tags = ["qgss26"]

if TOY_EXECUTOR_JOB_ID is not None:
    toy_job = service.job(TOY_EXECUTOR_JOB_ID)
    print(f"Re-using saved job: {TOY_EXECUTOR_JOB_ID}")
elif SUBMIT_TOY_EXEC_JOB:
    toy_job = executor.run(program)
    TOY_EXECUTOR_JOB_ID = toy_job.job_id()
    print(f"Submitted Executor job: {TOY_EXECUTOR_JOB_ID}")
else:
    print("Set SUBMIT_TOY_EXEC_JOB=True to submit a fresh executor job, "
          "or paste a saved job id into TOY_EXECUTOR_JOB_ID and re-run.")


In [ ]:
Check on the status of the job with the cell below:

In [ ]:
toy_job = service.job(TOY_EXECUTOR_JOB_ID)
print(f"{TOY_EXECUTOR_JOB_ID}  (status: {toy_job.status()})")

In [ ]:
Extract the results and look at the expectation value of $Z$ on qubit 0 and qubit 1 separately:

In [ ]:
if toy_job.status() == "DONE":
    toy_result = toy_job.result()
    data    = toy_result[0]
    meas    = data["c"]
    flips   = data["measurement_flips.c"]

    # Shape sanity check: layouts can vary across SDK versions.
    print(f"meas shape  : {meas.shape}")
    print(f"flips shape : {flips.shape}")

    corrected = np.bitwise_xor(meas, flips)
    z_vals    = 1 - 2 * corrected

    # Average over every leading axis (randomizations, shots, ...) so only the
    # final qubit axis remains. This is robust to the (randomizations, shots,
    # bits) vs (shots, bits) shape difference.
    z_means = z_vals.reshape(-1, z_vals.shape[-1]).mean(axis=0)

    print(f"\n<Z_0> = {z_means[0]:+.4f}")
    print(f"<Z_1> = {z_means[1]:+.4f}")
else:
    print(f"Not done yet (status={toy_job.status()}). Re-run cell when DONE.")


The ideal value is $\langle Z_i \rangle = +1$ on both qubits. This is because CZ acts trivially on $|00\rangle$, so the final state should remain $|00\rangle$. Any deviation from this state is hardware noise. In a typical run the deviation is at the percent level, with a size depending on backend, qubit layout, calibration at queue time, and shot budget. 

There are typically three contributions: __readout error__ on the two physical qubits, __idle decoherence__ during the single-qubit dressing rotations, and __residual stochastic Pauli error__ from the CZ. 

The deviation is also typically asymmetric between the two qubits because we transpiled at `optimization_level=0`: the qubit pair was chosen by index rather than by fidelity, so each qubit carries a different error budget. The dominant generator rates in 2.3.3 (order $10^{-3}$) give us an order-of-magnitude diagnostic, but the observed deviation also includes readout error, idle time, layout-dependent calibration, and accumulated box overhead, so the rates alone do not determine the gap.

Small per-layer rates like we saw in section 2.3.3 need to compound for mitigation effects to be visible. In section 2.6, we take the same pipeline to a deeper circuit (the 1D Ising mirror) and produces the `refs_to_noise_models` dict that Chapter 3's PNA and SLC will consume.
